# Lab 3: Finetuning Poseidon on Allen–Cahn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maximilianherde/ricam-foundation-models-for-pdes/blob/main/notebooks/03_poseidon_finetuning.ipynb)

**RICAM Training School, Linz, 9 October 2026**

Session 3 claimed that pretraining makes a model more sample-efficient on PDEs
it has not seen. In this lab you check this on the Allen–Cahn task of the
Poseidon paper, with the pretrained `Poseidon-T` checkpoint (21M parameters):

1. load the data and the checkpoint, and replace the embedding and recovery
   layers, since Allen–Cahn has one channel and pretraining used four;
2. finetune Poseidon-T on 32 trajectories, and train the same architecture
   from scratch on the same data;
3. repeat both for 4 and 16 trajectories to get sample-efficiency curves.

Our numbers will not match the paper's: we train for 6 epochs instead of up to
200, and use Poseidon-T instead of Poseidon-B (158M). The comparison between
the methods is what we can check.

Set the Colab runtime to a GPU: *Runtime → Change runtime type → T4 GPU*.

Herde et al., *Poseidon: Efficient Foundation Models for PDEs*, NeurIPS 2024
([arXiv:2405.19101](https://arxiv.org/abs/2405.19101)); code
[github.com/camlab-ethz/poseidon](https://github.com/camlab-ethz/poseidon)
(package `scOT`).

In [ ]:
N_TRAIN, N_TEST = 32, 240         # trajectories
EPOCHS, BATCH = 6, 8
SEED = 0

# training settings of the paper (SM C.1.1 and C.2)
OPT = {"finetuned": dict(lr=5e-5, lr_new=5e-4, warmup=0.0),
       "scratch":   dict(lr=5e-4, lr_new=None, warmup=0.05)}

## Setup

We clone the code of the paper, `scOT`, and put it on the Python path.

In [ ]:
import os, sys, math, time, inspect, subprocess
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from tqdm.auto import tqdm

SCOT_DIR = "/content/poseidon" if os.path.isdir("/content") else "./poseidon"
if not os.path.isdir(SCOT_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/camlab-ethz/poseidon", SCOT_DIR], check=True)
if SCOT_DIR not in sys.path:
    sys.path.insert(0, SCOT_DIR)

from scOT.model import ScOT, ScOTConfig, LayerNorm, ConditionalLayerNorm
from transformers.models.swinv2 import modeling_swinv2 as _sw

# compatibility with transformers >= 5.0
if "head_mask" not in inspect.signature(_sw.Swinv2Attention.forward).parameters:
    _attn_forward = _sw.Swinv2Attention.forward

    def _forward(self, hidden_states, attention_mask=None, head_mask=None, output_attentions=False):
        return _attn_forward(self, hidden_states, attention_mask, output_attentions=output_attentions)

    _sw.Swinv2Attention.forward = _forward

if not hasattr(ScOT, "get_head_mask"):
    ScOT.get_head_mask = lambda self, head_mask, n, *args: [None] * n

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## The problem and the data

The same dataset as in Labs 1 and 2: the Allen–Cahn equation
$u_t = \Delta u - \varepsilon^2 u(u^2-1)$, $\varepsilon = 220$, on the unit
square, 20 snapshots on a $128\times128$ grid. Training data are the first 32
trajectories of `solution_0.nc`; the test set is the paper's, the last 240
trajectories of `solution_3.nc`.

**Which snapshots.** We follow the paper's protocol for this dataset: only
snapshots $0, 2, 4, \dots, 14$ are used, and the model is evaluated on the map
from snapshot 0 to snapshot 14. So we download only these snapshots.

**The operator.** The model maps $u(t_1)$ and the lead time $t_2 - t_1$ to
$u(t_2)$. The lead time is divided by 19 (the index of the last snapshot)
before it enters the model.

Fields are standardised with Poseidon's mean and standard deviation for this
dataset, and errors are reported on the standardised fields, as in the paper.
They could equally be reported on the physical field; the values differ
slightly.

In [ ]:
try:
    import fsspec, h5py
except ImportError:
    !pip install -q fsspec h5py
    import fsspec, h5py

URL = "https://huggingface.co/datasets/camlab-ethz/ACE/resolve/main/solution_{}.nc"
MEAN, STD, TIME_SCALE = 0.002484262, 0.65351176, 19.0      # scOT's constants for ACE
TRAIN_SNAPSHOTS = slice(0, 15, 2)       # 0, 2, ..., 14
TEST_SNAPSHOTS = slice(0, 15, 14)       # 0 and 14


def load_ace(file, start, stop, snapshots, cache="ace_cache"):
    path = os.path.join(cache, f"ace{file}_{start}_{stop}_{snapshots.start}_{snapshots.step}.npy")
    if not os.path.exists(path):
        os.makedirs(cache, exist_ok=True)
        with fsspec.open(URL.format(file), "rb", block_size=2**22) as f, h5py.File(f, "r") as h:
            np.save(path, h["solution"][start:stop, snapshots])
    return (np.load(path) - MEAN) / STD


def load_train(n):
    return load_ace(0, 0, n, TRAIN_SNAPSHOTS)


def load_test(n=N_TEST):
    return load_ace(3, 3750 - n, 3750, TEST_SNAPSHOTS)

In [ ]:
train_data = load_train(N_TRAIN)
test_data = load_test()
print("train", train_data.shape, "  test", test_data.shape)

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, j in zip(axes, [0, 2, 4, 7]):
    ax.imshow(train_data[0, j], cmap="gist_ncar", vmin=-1.6, vmax=1.6)
    ax.set_title(f"snapshot {2 * j}"); ax.axis("off")
plt.tight_layout(); plt.show()

### All2all training

Poseidon is trained on every pair $(t_1, t_2)$ with $t_1 \le t_2$ from the
same trajectory. With 8 snapshots that is 36 pairs per trajectory, so 32
trajectories give $32 \times 36 = 1152$ training examples. The sampler draws
(trajectory, $t_1$, $t_2$) uniformly at random.

In [ ]:
class All2All:
    def __init__(self, data, seed=0):
        self.data = torch.from_numpy(np.ascontiguousarray(data, dtype=np.float32))  # (N, 8, H, W)
        n_t = data.shape[1]
        self.pairs = np.array([(i, j) for i in range(n_t) for j in range(i, n_t)])
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.data) * len(self.pairs)

    def batch(self, bs):
        i = self.rng.integers(0, len(self.data), bs)
        i1, i2 = self.pairs[self.rng.integers(0, len(self.pairs), bs)].T
        x = self.data[i, i1].unsqueeze(1).to(DEVICE)
        y = self.data[i, i2].unsqueeze(1).to(DEVICE)
        lead = torch.from_numpy(2.0 * (i2 - i1) / TIME_SCALE).float().to(DEVICE)  # stored index = snapshot / 2
        return x, y, lead

In [ ]:
sampler = All2All(train_data, seed=SEED)
print(f"{N_TRAIN} trajectories x {len(sampler.pairs)} pairs = {len(sampler)} training examples")

## Poseidon-T on a non-native task

Poseidon was pretrained on compressible and incompressible flow data with four
channels. Allen–Cahn has one input and one output channel, so the patch
embedding and the patch recovery have the wrong shape and are initialised
anew. Everything in between, the backbone, is loaded from the checkpoint. With
`ignore_mismatched_sizes=True`, `from_pretrained` loads every tensor whose
shape matches and re-initialises the others.

**How the lead time enters the model.** Every layer norm in Poseidon is
time-conditioned: after normalising, it multiplies by $a(t)$ and adds $b(t)$,
where $a$ and $b$ are linear functions of the lead time $t$
(`ConditionalLayerNorm` in `scOT/model.py`). These parameters are pretrained
too.

### Exercise 1: Poseidon-T for one channel

`poseidon_t_config(num_channels, num_out_channels)` returns the architecture
of Poseidon-T for a given number of input and output channels. Write
`build_model(pretrained)`:

- with `pretrained=True`, load the checkpoint `camlab-ethz/Poseidon-T` into a
  model with one input and one output channel, so that every pretrained tensor
  of matching shape is loaded and the others are initialised anew;
- with `pretrained=False`, return the same architecture randomly initialised
  (seeded with `seed`).

The cell after it counts how many tensors came from the checkpoint. Each
exercise has a hint you can open if you are stuck.

In [ ]:
def poseidon_t_config(num_channels, num_out_channels, resolution=128):
    # the published architecture of Poseidon-T
    return ScOTConfig(
        image_size=resolution, patch_size=4,
        num_channels=num_channels, num_out_channels=num_out_channels,
        embed_dim=48, depths=[4, 4, 4, 4], num_heads=[3, 6, 12, 24],
        skip_connections=[2, 2, 2, 0], window_size=16, mlp_ratio=4.0, qkv_bias=True,
        hidden_dropout_prob=0.0, attention_probs_dropout_prob=0.0, drop_path_rate=0.0,
        hidden_act="gelu", use_absolute_embeddings=False, initializer_range=0.02,
        layer_norm_eps=1e-5, p=1, channel_slice_list_normalized_loss=[0, num_out_channels],
        residual_model="convnext", use_conditioning=True, learn_residual=False)

In [ ]:
def build_model(pretrained, seed=SEED):
    raise NotImplementedError("Exercise 1")

<details><summary><b>Hint</b></summary>

`ScOT.from_pretrained(name, config=..., ignore_mismatched_sizes=True)` loads a
checkpoint into a model built from `config`; tensors whose shape does not match
are initialised anew. Without a checkpoint, `ScOT(config)` builds a randomly
initialised model.

</details>

In [ ]:
model_ft = build_model(pretrained=True)
native = ScOT.from_pretrained("camlab-ethz/Poseidon-T")       # the original 4-channel model
native_sd = native.state_dict()

new = [k for k, v in model_ft.state_dict().items()
       if not (k in native_sd and native_sd[k].shape == v.shape and torch.equal(native_sd[k], v))]
n_total = sum(p.numel() for p in model_ft.parameters())
print(f"{len(model_ft.state_dict()) - len(new)} of {len(model_ft.state_dict())} tensors loaded from the checkpoint")
print("re-initialised:", *new, sep="\n  ")
print(f"\n{n_total:,} parameters")
del native, native_sd

## Training settings

We use the settings of the paper (SM C.1.1 for finetuning, SM C.2 for scOT
trained from scratch), except for the number of epochs and the batch size:

- **finetuning**: embedding and recovery with learning rate $5\cdot10^{-4}$,
  the time-conditioned layer norms with $5\cdot10^{-4}$, all other parameters
  with $5\cdot10^{-5}$. Cosine decay of the learning rate.
- **from scratch**: learning rate $5\cdot10^{-4}$ for all parameters, cosine
  decay after a linear warmup over the first 5% of the steps (20 of 400 epochs
  in the paper).

In both cases AdamW with weight decay $10^{-6}$, no weight decay on layer norm
parameters and biases, and gradient clipping at 5. The paper trains for 200
(finetuning) or 400 epochs at batch size 40; we train for 6 epochs at batch
size 8.

### Exercise 2: parameter groups

Write `make_optimizer(model, lr, lr_new=None, weight_decay=1e-6)`, which returns
an AdamW optimizer with the parameter groups of `scOT/trainer.py`. Go through
the parameters by name and put each into the first group that applies:

1. only if `lr_new` is given: names containing `embeddings` or
   `patch_recovery`: learning rate `lr_new`, weight decay `weight_decay`;
2. parameters that are not inside a layer norm (`torch.nn.LayerNorm`,
   `LayerNorm` or `ConditionalLayerNorm`) and whose name does not contain
   `bias`: learning rate `lr`, weight decay `weight_decay`;
3. only if `lr_new` is given: parameters inside a `ConditionalLayerNorm`:
   learning rate `lr_new`, no weight decay;
4. all others: learning rate `lr`, no weight decay.

Leave out empty groups. The cell after it checks the group sizes.

In [ ]:
def make_optimizer(model, lr, lr_new=None, weight_decay=1e-6):
    raise NotImplementedError("Exercise 2")

<details><summary><b>Hint</b></summary>

`model.named_modules()` gives `(module_name, module)`; with
`isinstance(module, ...)` you find the layer norms, and the full names of their
parameters are `f"{module_name}.{p}"` for `p, _ in module.named_parameters()`.
Compare these names with the ones from `model.named_parameters()`.

</details>

In [ ]:
# Check: (number of tensors, learning rate, weight decay) per group
def summary(opt):
    return sorted((len(g["params"]), g["lr"], g["weight_decay"]) for g in opt.param_groups)


assert summary(make_optimizer(model_ft, 5e-5, 5e-4)) == \
    [(9, 5e-4, 1e-6), (257, 5e-5, 1e-6), (274, 5e-5, 0.0), (304, 5e-4, 0.0)]
assert summary(make_optimizer(model_ft, 5e-4)) == [(260, 5e-4, 1e-6), (584, 5e-4, 0.0)]
print("ok")

In [ ]:
class LiveLoss:
    # Plots the training loss while training. Call update(loss) every step and
    # close() at the end; the plot shows the mean over every `every` steps.
    def __init__(self, title="", every=20):
        self.every, self.steps, self.values = every, [], []
        self.window, self.n = 0.0, 0
        self.fig, self.ax = plt.subplots(figsize=(5, 2.6))
        self.title = title
        self.handle = display(self.fig, display_id=True)
        plt.close(self.fig)

    def update(self, loss):
        self.window = self.window + loss.detach()
        self.n += 1
        if self.n % self.every == 0:
            self.steps.append(self.n)
            self.values.append(self.window.item() / self.every)
            self.window = 0.0
            self.draw()

    def draw(self):
        self.ax.clear()
        self.ax.semilogy(self.steps, self.values)
        self.ax.set_xlabel("step"); self.ax.set_ylabel("training loss")
        self.ax.set_title(self.title, fontsize=9)
        self.fig.tight_layout()
        self.handle.update(self.fig)

    def close(self):
        if self.steps:
            self.draw()



def train(model, sampler, steps, method, batch=BATCH, desc=""):
    # method: "finetuned" or "scratch", see OPT
    model.to(DEVICE).train()
    o = OPT[method]
    opt = make_optimizer(model, o["lr"], o["lr_new"])
    base = [g["lr"] for g in opt.param_groups]
    n_warm = int(o["warmup"] * steps)
    live = LiveLoss(desc)
    for s in tqdm(range(steps), desc=desc):
        f = (s + 1) / n_warm if s < n_warm else \
            0.5 * (1 + math.cos(math.pi * (s - n_warm) / (steps - n_warm)))
        for g, b in zip(opt.param_groups, base):
            g["lr"] = b * f
        x, y, lead = sampler.batch(batch)
        loss = model(pixel_values=x, time=lead, labels=y, return_dict=True).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        live.update(loss)
    live.close()
    return model


@torch.no_grad()
def evaluate(model, test, batch=16):
    # relative L1 error (%) per test trajectory, snapshot 0 -> 14
    model.to(DEVICE).eval()
    preds = []
    for k in range(0, len(test), batch):
        x = torch.from_numpy(test[k:k + batch, 0]).float().unsqueeze(1).to(DEVICE)
        lead = torch.full((len(x),), 14 / TIME_SCALE, device=DEVICE)
        preds.append(model(pixel_values=x, time=lead, return_dict=True).output.cpu().numpy())
    pred = np.concatenate(preds)[:, 0]
    true = test[:, 1]
    errs = 100 * np.abs(pred - true).sum(axis=(1, 2)) / np.abs(true).sum(axis=(1, 2))
    return errs, pred


def steps_for(n_traj, epochs=EPOCHS, batch=BATCH):
    return round(epochs * n_traj * 36 / batch)

## Finetuning against training from scratch

Both models see the same 32 trajectories, for the same number of steps
(6 epochs at batch size 8), with the same schedule. They differ only in the
initial weights and the learning rates. We report the median relative $L^1$
error over the 240 test trajectories, as the paper does.

In [ ]:
STEPS = steps_for(N_TRAIN)
model_ft = train(model_ft, sampler, STEPS, "finetuned", desc="finetuned")
errs_ft, pred_ft = evaluate(model_ft, test_data)

model_sc = train(build_model(pretrained=False), All2All(train_data, seed=SEED), STEPS,
                 "scratch", desc="from scratch")
errs_sc, pred_sc = evaluate(model_sc, test_data)

print(f"{'':24} {'median':>8} {'mean':>8} {'90th pct':>9}")
for name, e in [("Poseidon-T, finetuned", errs_ft), ("from scratch", errs_sc)]:
    print(f"{name:24} {np.median(e):7.2f}% {e.mean():7.2f}% {np.percentile(e, 90):8.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
bins = np.linspace(0, max(errs_ft.max(), errs_sc.max()) * 1.05, 30)
ax.hist(errs_sc, bins=bins, alpha=0.7, label="from scratch")
ax.hist(errs_ft, bins=bins, alpha=0.7, label="Poseidon-T, finetuned")
ax.set_xlabel("relative $L^1$ error, snapshot 0 → 14 (%)"); ax.set_ylabel("test trajectories")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

k = int(np.argsort(errs_ft)[len(errs_ft) // 2])          # a test trajectory of median difficulty
truth = test_data[k, 1]
panels = [("input, snapshot 0", test_data[k, 0]), ("true, snapshot 14", truth),
          (f"finetuned ({errs_ft[k]:.1f}%)", pred_ft[k]), (f"from scratch ({errs_sc[k]:.1f}%)", pred_sc[k])]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (t, f) in zip(axes, panels):
    ax.imshow(f, cmap="gist_ncar", vmin=-1.6, vmax=1.6); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

### The prediction over time

For the last test trajectory, both models predict snapshots $0, 2, \dots, 14$
from snapshot 0, each with a single call at the corresponding lead time. This
is how the paper evaluates this dataset (direct evaluation, SM Table 6), as
opposed to an autoregressive rollout. Shown in physical units:

In [ ]:
def movie(fields, titles, labels, ranges=None):
    # fields: list of arrays of shape (frames, H, W), shown side by side;
    # labels: one title per frame
    n = len(fields)
    ranges = ranges or [(-1, 1)] * n
    fig, axes = plt.subplots(1, n, figsize=(2.7 * n, 2.9))
    axes = np.atleast_1d(axes)
    ims = []
    for ax, f, t, (lo, hi) in zip(axes, fields, titles, ranges):
        ims.append(ax.imshow(f[0], cmap="gist_ncar", vmin=lo, vmax=hi))
        ax.set_title(t, fontsize=9); ax.axis("off")
    label = fig.suptitle(labels[0])

    def frame(k):
        for im, f in zip(ims, fields):
            im.set_data(f[k])
        label.set_text(labels[k])
        return ims

    anim = animation.FuncAnimation(fig, frame, frames=len(fields[0]), interval=400)
    plt.close(fig)
    return HTML(anim.to_jshtml())


@torch.no_grad()
def predict_from_start(model, u0, leads):
    # u0: (H, W) standardised; one prediction per lead time (in snapshots)
    model.eval()
    x = torch.from_numpy(u0).float().to(DEVICE).expand(len(leads), 1, -1, -1)
    t = torch.tensor(leads, dtype=torch.float32, device=DEVICE) / TIME_SCALE
    out = model(pixel_values=x, time=t, return_dict=True).output
    return out.cpu().numpy()[:, 0]


traj = load_ace(3, 3749, 3750, TRAIN_SNAPSHOTS)[0]          # snapshots 0, 2, ..., 14
leads = list(range(0, 15, 2))
to_u = lambda v: v * STD + MEAN
true_u = to_u(traj)
ft_u = to_u(predict_from_start(model_ft, traj[0], leads))
sc_u = to_u(predict_from_start(model_sc, traj[0], leads))

movie([true_u, ft_u, sc_u, np.abs(ft_u - true_u), np.abs(sc_u - true_u)],
      ["true", "finetuned", "from scratch", "|error| finetuned", "|error| from scratch"],
      [f"snapshot {l}" for l in leads],
      ranges=[(-1, 1)] * 3 + [(0, 0.5)] * 2)

## The sample-efficiency curve

One dataset size is not enough to talk about sample efficiency. We repeat the
two trainings for 4 and 16 trajectories; the runs at 32 trajectories are
the ones above. Every run gets 6 epochs, so the number of steps grows with the
dataset size. Each point is a single run; differences between runs with
different random seeds are not measured here.

The **efficiency gain** asks how many trajectories the model trained from
scratch needs to reach the error the finetuned model reaches with $N$. On the
log–log plot it is the horizontal distance between the two curves.

In [ ]:
N_CURVE = [4, 16, 32]
curve = {"finetuned": {}, "from scratch": {}}
curve["finetuned"][N_TRAIN] = np.median(errs_ft)
curve["from scratch"][N_TRAIN] = np.median(errs_sc)

for n in N_CURVE:
    if n == N_TRAIN:
        continue
    data = load_train(n)
    for name, pretrained, method in [("finetuned", True, "finetuned"),
                                     ("from scratch", False, "scratch")]:
        m = build_model(pretrained)
        m = train(m, All2All(data, seed=SEED), steps_for(n), method, desc=f"N={n}, {name}")
        curve[name][n] = np.median(evaluate(m, test_data)[0])
        del m
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        elif DEVICE == "mps":
            torch.mps.empty_cache()

print(f"{'N':>4} " + " ".join(f"{name:>16}" for name in curve))
for n in N_CURVE:
    print(f"{n:4d} " + " ".join(f"{curve[name][n]:15.2f}%" for name in curve))

In [ ]:
ft = np.array([curve["finetuned"][n] for n in N_CURVE])
sc = np.array([curve["from scratch"][n] for n in N_CURVE])
logN, logsc = np.log(N_CURVE), np.log(sc)

print(f"{'N':>4} {'ratio of errors':>16} {'N from scratch needs':>21} {'efficiency gain':>16}")
for n, e_ft, e_sc in zip(N_CURVE, ft, sc):
    if sc.min() <= e_ft <= sc.max():       # interpolate in log-log, do not extrapolate
        need = float(np.exp(np.interp(np.log(e_ft), logsc[::-1], logN[::-1])))
        print(f"{n:4d} {e_sc / e_ft:15.1f}x {need:21.0f} {need / n:15.1f}x")
    else:
        print(f"{n:4d} {e_sc / e_ft:15.1f}x {'> ' + str(N_CURVE[-1]):>21}")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for name in curve:
    ax.plot(N_CURVE, [curve[name][n] for n in N_CURVE], "-o", label=name)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xticks(N_CURVE); ax.set_xticklabels(N_CURVE)
ax.set_xlabel("training trajectories $N$"); ax.set_ylabel("median relative $L^1$ error (%)")
ax.grid(True, which="both", lw=0.4, alpha=0.4); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## Comparison with the paper

The paper reports for Allen–Cahn (SM D.4.2 and Figure 19), with Poseidon-B
trained to convergence:

| | $N$ | median relative $L^1$ |
|---|---|---|
| Poseidon-B, finetuned | 32 | 1.4% |
| FNO | 128 | 3.7% |

Note that the FNO number is at 128 trajectories, not 32.

In [ ]:
print(f"{'':28} {'N':>4} {'this lab':>9} {'paper':>8}")
for name, ours, paper in [("Poseidon-T, finetuned", np.median(errs_ft), "1.4%"),
                          ("from scratch", np.median(errs_sc), "--")]:
    print(f"{name:28} {N_TRAIN:4d} {ours:8.2f}% {paper:>8}")

## Something to try

Finetune Poseidon-T on a time-independent problem: **Poisson-Gauss** from the
paper (SM B.2.14), $-\Delta u = f$ on the unit square, where the source $f$ is
a sum of Gaussians. The task is to map $f$ to $u$.

- The data is `Poisson-Gauss.nc` in `camlab-ethz/Poisson-Gauss` on Hugging
  Face, with variables `source` and `solution`, each of shape
  `(20000, 128, 128)`. As for Allen–Cahn, the test set is the last 240
  samples.
- Input and output are standardised separately; the constants are in
  `scOT/problems/elliptic/poisson.py`.
- There is no time. `scOT` still uses the time-conditioned model and passes a
  constant lead time of 1 (`TimeWrapper` in `scOT/problems/base.py`).

How does the finetuned model compare with the same architecture trained from
scratch, and does the gap look like the one for Allen–Cahn?